In [54]:
import numpy as np
import pandas as pd
import re
import sys
from frechetdist import frdist
from concurrent.futures import ThreadPoolExecutor
from typing import List, Tuple

In [55]:
class AnySecDataProcessor:
    def __init__(self, study_data):
        self.data = study_data

    def extract_agvname_number(self):
        """Extract the number from 'AGV[x]' in AGVname."""
        def extract_number(string):
            numbers = re.findall(r'\d+', string)
            return int(numbers[0]) if numbers else None

        self.data['AGVname'] = self.data['AGVname'].apply(extract_number)

    def swap_agvname_values(self):
        """Swap values 1 and 2 in the 'AGVname' column."""
        self.data['AGVname'].replace([2, 1], ['first', 'second'], inplace=True)
        self.data['AGVname'].replace(['first', 'second'], [1, 2], inplace=True)

    def apply_zero_padding_to_pid(self):
        """Apply zero-padding to the 'PID' column to ensure consistency."""
        self.data['PID'] = self.data['PID'].apply(lambda x: str(x).zfill(3))

    def drop_pid_rows(self):
        """Drop PID = 5, 18, 23, 25 due to insufficient data."""
        self.data = self.data.drop(
            self.data[self.data['PID'].isin([5, 18, 23, 25])].index
        )    

    def calculate_gaze_on_agv(self):
        """Calculate and add 'Gaze_on_AGV' to Per Interaction Data."""
        result_dict_for_gaze_on_agv = {}
        grouped_for_gaze = self.data.groupby(['PID', 'AGVname', 'DRate'])

        for name, group in grouped_for_gaze:
            true_count = group['Gaze_on_AGV'].sum()
            total_count = group['Gaze_on_AGV'].count()
            fraction = round(true_count / total_count, 3)
            result_dict_for_gaze_on_agv[name] = fraction

        self.data['Gaze_on_AGV'] = self.data.apply(
            lambda row: result_dict_for_gaze_on_agv.get((row['PID'], row['AGVname'], row['DRate']), None), axis=1
        )

    def calculate_user_relative_speed(self):
        """Calculate 'User_Relative_Speed' based on changes in User_X, User_Y, and User_Z without keeping intermediate columns."""
        grouped_data = self.data.groupby(['PID', 'AGVname', 'DRate'])
        
        self.data['User_Relative_Speed'] = grouped_data.apply(
            lambda group: np.sqrt(
                group['User_X'].diff().fillna(0)**2 +
                group['User_Y'].diff().fillna(0)**2 +
                group['User_Z'].diff().fillna(0)**2
            ) / 0.1
        ).reset_index(level=[0, 1, 2], drop=True)

    def calculate_agv_relative_speed(self):
        """Calculate 'AGV_Relative_Speed' based on changes in AGV_X, AGV_Y, and AGV_Z without keeping intermediate columns."""
        grouped_data = self.data.groupby(['PID', 'AGVname', 'DRate'])
        
        self.data['AGV_Relative_Speed'] = grouped_data.apply(
            lambda group: np.sqrt(
                group['AGV_X'].diff().fillna(0)**2 +
                group['AGV_Y'].diff().fillna(0)**2 +
                group['AGV_Z'].diff().fillna(0)**2
            ) / 0.1
        ).reset_index(level=[0, 1, 2], drop=True)

    def calculate_agv_user_distance(self):
        """Calculate 'AGV_User_distance' as the Euclidean distance between AGV and User in 3D space."""
        grouped_data = self.data.groupby(['PID', 'AGVname', 'DRate'])
        
        self.data['AGV_User_distance'] = grouped_data.apply(
            lambda group: np.sqrt(
                (group['User_X'] - group['AGV_X']) ** 2 +
                (group['User_Y'] - group['AGV_Y']) ** 2 +
                (group['User_Z'] - group['AGV_Z']) ** 2
            )
        ).reset_index(level=[0, 1, 2], drop=True)
        
    def calculate_new_timestamp(self):
        """Calculate new timestamps based on quantiles and update the Timestamp column."""
        def new_timestamp(row, suffix):
            # Convert the original timestamp to datetime
            original_time = pd.to_datetime(row['Timestamp'], format='%H:%M:%S')
            # Create the new timestamp with the appropriate suffix
            new_time = f"{original_time.strftime('%H:%M:%S')}{suffix}"
            return new_time
    
        # Convert the Timestamp column to datetime format
        self.data['Timestamp'] = pd.to_datetime(self.data['Timestamp'], format='%H:%M:%S')
    
        # Define suffixes for different quantile groups
        suffix_map = {
            1: ".0",   # For quantile 1
            2: ".1",   # For quantile 2
            3: ".2",   # For quantile 3
            4: ".3",  # For quantile 4
            5: ".4",  # For quantile 5
            6: ".5",  # For quantile 6
            7: ".6",  # For quantile 7
            8: ".7",  # For quantile 8
            9: ".8",  # For quantile 9
            10: ".9"  # For quantile 10
        }
    
        # Apply new timestamps based on quantiles
        for quantile in suffix_map.keys():
            # Filter the data for the current quantile
            quantile_group = self.data[self.data['quantile'] == quantile]
            if not quantile_group.empty:
                # Update timestamps for the filtered group
                self.data.loc[self.data['quantile'] == quantile, 'Timestamp'] = quantile_group.apply(
                    lambda row: new_timestamp(row, suffix_map[quantile]), axis=1
                )
    
        # Drop the quantile column if it exists
        if 'quantile' in self.data.columns:
            self.data.drop(columns=['quantile'], inplace=True)

    def process(self):
        """Run all processing steps."""
        self.extract_agvname_number()
        self.swap_agvname_values()
        self.drop_pid_rows()
        self.apply_zero_padding_to_pid()
        self.calculate_gaze_on_agv()
        self.calculate_user_relative_speed()
        self.calculate_agv_relative_speed()
        self.calculate_agv_user_distance()
        self.calculate_new_timestamp()  # Call the new method here
        
        return self.data

In [65]:
class PerInteractionDataProcessor:
    def __init__(self, per_interaction_data, study_data):
        # Initialize instance variables without 'self.self'
        self.per_interaction_data = per_interaction_data
        self.study_data = study_data  # Make sure you're initializing this properly

    def swap_agvname_values(self):
        """Swap values 1 and 2 in the 'AGVname' column."""
        self.per_interaction_data['AGVname'].replace([2, 1], ['first', 'second'], inplace=True)
        self.per_interaction_data['AGVname'].replace(['first', 'second'], [1, 2], inplace=True)

    def drop_pid_rows(self):
        """Drop PID = 5, 18, 23, 25 due to insufficient data."""
        self.per_interaction_data = self.per_interaction_data.drop(
            self.per_interaction_data[self.per_interaction_data['PID'].isin([5, 18, 23, 25])].index
        )

    def apply_zero_padding_to_pid(self):
        """Apply zero-padding to the 'PID' column to ensure consistency."""
        self.per_interaction_data['PID'] = self.per_interaction_data['PID'].apply(lambda x: str(x).zfill(3))

    def calculate_gaze_on_agv(self):
        """Calculate and add 'Gaze_on_AGV' to Per Interaction Data."""
        if 'Gaze_on_AGV' not in self.study_data.columns:
            print("Column 'Gaze_on_AGV' does not exist in study_data.")
            return  # Or handle the case as appropriate
        
        result_dict_for_gaze_on_agv = {}
        grouped_for_gaze = self.study_data.groupby(['PID', 'AGVname', 'DRate'])
    
        for name, group in grouped_for_gaze:
            true_count = group['Gaze_on_AGV'].sum()
            total_count = group['Gaze_on_AGV'].count()
            fraction = round(true_count / total_count, 3)
            result_dict_for_gaze_on_agv[name] = fraction
    
        self.per_interaction_data['Gaze_on_AGV'] = self.per_interaction_data.apply(
            lambda row: result_dict_for_gaze_on_agv.get((row['PID'], row['AGVname'], row['DRate']), None), axis=1
        )

    def calculate_user_relative_speed(self):
        """Calculate 'User_Relative_Speed' based on changes in User_X, User_Y, and User_Z without keeping intermediate columns."""
        grouped_data = self.per_interaction_data.groupby(['PID', 'AGVname', 'DRate'])

        self.per_interaction_data['User_Relative_Speed'] = grouped_data.apply(
            lambda group: np.sqrt(
                group['User_X'].diff().fillna(0)**2 +
                group['User_Y'].diff().fillna(0)**2 +
                group['User_Z'].diff().fillna(0)**2
            ) / 0.1
        ).reset_index(level=[0, 1, 2], drop=True)

    def calculate_agv_relative_speed(self):
        """Calculate 'AGV_Relative_Speed' based on changes in AGV_X, AGV_Y, and AGV_Z without keeping intermediate columns."""
        grouped_data = self.per_interaction_data.groupby(['PID', 'AGVname', 'DRate'])

        self.per_interaction_data['AGV_Relative_Speed'] = grouped_data.apply(
            lambda group: np.sqrt(
                group['AGV_X'].diff().fillna(0)**2 +
                group['AGV_Y'].diff().fillna(0)**2 +
                group['AGV_Z'].diff().fillna(0)**2
            ) / 0.1
        ).reset_index(level=[0, 1, 2], drop=True)

    def calculate_agv_user_distance(self):
        """Calculate 'AGV_User_distance' as the Euclidean distance between AGV and User in 3D space."""
        grouped_data = self.per_interaction_data.groupby(['PID', 'AGVname', 'DRate'])

        self.per_interaction_data['AGV_User_distance'] = grouped_data.apply(
            lambda group: np.sqrt(
                (group['User_X'] - group['AGV_X']) ** 2 +
                (group['User_Y'] - group['AGV_Y']) ** 2 +
                (group['User_Z'] - group['AGV_Z']) ** 2
            )
        ).reset_index(level=[0, 1, 2], drop=True)

    def map_user_trajectory(self):
        """Map 'User_Trajectory' based on 'AGVname'."""
        user_trajectory_mapping = {
            1: 'Straight', 2: 'Diagonal', 3: 'Diagonal', 4: 'Straight', 5: 'Straight',
            6: 'Diagonal', 7: 'Straight', 8: 'Diagonal', 9: 'Diagonal', 10: 'Straight',
            11: 'Diagonal', 12: 'Straight', 13: 'Straight', 14: 'Diagonal', 15: 'Straight',
            16: 'Diagonal'
        }
        # Map the values based on 'AGVname'
        self.per_interaction_data['User_Trajectory'] = self.per_interaction_data['AGVname'].map(user_trajectory_mapping)

    def map_agv_approaching(self):
        """Map 'AGV_Approaching' based on 'AGVname'."""
        agv_approaching_mapping = {
            1: 'South', 2: 'North', 3: 'South', 4: 'Northeast', 5: 'Northwest', 
            6: 'Northwest', 7: 'East', 8: 'Southwest', 9: 'Northeast', 10: 'West', 
            11: 'East', 12: 'Southeast', 13: 'Southwest', 14: 'Southeast', 15: 'North', 
            16: 'West'
        }
        # Map the values based on 'AGVname'
        self.per_interaction_data['AGV_Approaching'] = self.per_interaction_data['AGVname'].map(agv_approaching_mapping)

    def create_agv_user_combination(self):
        """Create a new column 'AGV_User_Combination' by combining 'AGV_Approaching' and 'User_Trajectory'."""
        self.per_interaction_data['AGV_User_Combination'] = (
            self.per_interaction_data['AGV_Approaching'].astype(str) + 
            ' - ' + 
            self.per_interaction_data['User_Trajectory'].astype(str)
        )

    def map_agv_path_complexity(self):
        """Map 'AGV_User_Combination' to 'AGV_Path_Complexity' based on predefined complexity mappings."""
        complexity_mapping = {
            'North - Diagonal': 'Complex',
            'South - Straight': 'Straight',
            'South - Diagonal': 'Complex',
            'Northeast - Straight': 'Complex',
            'Northwest - Straight': 'Complex',
            'Northwest - Diagonal': 'Complex',
            'East - Straight': 'Straight',
            'Southwest - Diagonal': 'Straight',
            'Northeast - Diagonal': 'Straight',
            'West - Straight': 'Straight',
            'East - Diagonal': 'Complex',
            'Southeast - Straight': 'Complex',
            'Southwest - Straight': 'Complex',
            'Southeast - Diagonal': 'Straight',
            'North - Straight': 'Straight',
            'West - Diagonal': 'Complex',
        }
        
        self.per_interaction_data['AGV_Path_Complexity'] = self.per_interaction_data['AGV_User_Combination'].map(complexity_mapping)

    def calculate_trust_before(self, high_first_low_last, low_first_high_last):
        """Calculate the 'Trust_before' column for the DataFrame based on PID and DRate."""
        
        # Ensure PID is of the correct type
        self.per_interaction_data['PID'] = self.per_interaction_data['PID'].astype(int).astype(str).str.zfill(3)
    
        # Group the data by PID
        grouped_for_trust_before = self.per_interaction_data.groupby('PID')
    
        # Iterate over each PID group
        for pid, group in grouped_for_trust_before:
            group_high = group[group['DRate'] == 'High']
            group_low = group[group['DRate'] == 'Low']
            
            # Check which list the PID belongs to
            if pid in high_first_low_last:
                if not group_high.empty:
                    self.per_interaction_data.loc[group_high.index, 'Trust_before'] = group_high['Trust'].shift(1)
                if not group_low.empty:
                    # Check for None or NaN before dividing
                    if pd.notna(group_low['Trust1'].iloc[0]):
                        self.per_interaction_data.loc[group_low.index[0], 'Trust_before'] = group_low['Trust1'].iloc[0] / 10
                    self.per_interaction_data.loc[group_low.index[1:], 'Trust_before'] = group_low['Trust'].shift(1)
    
            elif pid in low_first_high_last:
                if not group_low.empty:
                    self.per_interaction_data.loc[group_low.index, 'Trust_before'] = group_low['Trust'].shift(1)
                if not group_high.empty:
                    # Check for None or NaN before dividing
                    if pd.notna(group_high['Trust1'].iloc[0]):
                        self.per_interaction_data.loc[group_high.index[0], 'Trust_before'] = group_high['Trust1'].iloc[0] / 10
                    self.per_interaction_data.loc[group_high.index[1:], 'Trust_before'] = group_high['Trust'].shift(1)


    def process_cross_first(self, cross_first):
        """Process the cross_first data and apply necessary transformations or logic."""
        
        # Ensure PID is of the correct type
        cross_first['PID'] = cross_first['PID'].astype(int).astype(str).str.zfill(3)
        
        # Merge the two DataFrames on the common columns
        merged_data = self.per_interaction_data.merge(
            cross_first[['PID', 'DRate', 'AGVname', 'cross_first']],
            on=['PID', 'DRate', 'AGVname'],
            how='left'
        )
        
        # Update the Cross_First column in per_interaction_data
        self.per_interaction_data['Cross_First'] = merged_data['cross_first']

    def calculate_relative_speed(self):
        """
        Calculate User and AGV relative speeds for each group based on PID, DRate, and AGVname.
    
        Parameters:
        interaction_data (pd.DataFrame): DataFrame containing interaction times with StartTime and EndTime.
        study_data (pd.DataFrame): The study_data containing user and AGV coordinates, grouped based on PID, DRate, and AGVname.
    
        Returns:
        pd.DataFrame: DataFrame with User_Relative_Speed and AGV_Relative_Speed for each group.
        """
    
        # Group study_data based on PID, DRate, and AGVname
        grouped_study_data = self.study_data.groupby(['PID', 'DRate', 'AGVname'])

        self.per_interaction_data['StartTime'] = pd.to_datetime(self.per_interaction_data['StartTime'], errors='coerce', format='%H:%M:%S')
        self.per_interaction_data['EndTime'] = pd.to_datetime(self.per_interaction_data['EndTime'], errors='coerce', format='%H:%M:%S')
        
        # Iterate through each group and calculate relative speed
        for (pid, drate, agvname), group in grouped_study_data:
            
            # Get the first and last entries of the group (coordinates)
            first_row = group.iloc[0]
            last_row = group.iloc[-1]
            
            # Filter per_interaction_data for the same group
            interaction_group = self.per_interaction_data[(self.per_interaction_data['PID'] == pid) &
                                                          (self.per_interaction_data['DRate'] == drate) &
                                                          (self.per_interaction_data['AGVname'] == agvname)]
            
            if interaction_group.empty:
                # If there is no matching group in per_interaction_data, skip
                continue
            
            # Get StartTime and EndTime from per_interaction_data
            start_time = pd.to_datetime(interaction_group['StartTime'].values[0])
            end_time = pd.to_datetime(interaction_group['EndTime'].values[0])
            
            # Calculate the time difference in seconds
            time_diff = (end_time - start_time).total_seconds()
            
            if time_diff == 0:
                # If the time difference is zero, set NaN for both relative speeds
                self.per_interaction_data.loc[(self.per_interaction_data['PID'] == pid) & 
                                              (self.per_interaction_data['DRate'] == drate) &
                                              (self.per_interaction_data['AGVname'] == agvname),
                                              ['User_Relative_Speed', 'AGV_Relative_Speed']] = [np.nan, np.nan]
                continue
            
            # Get the difference in coordinates (assuming columns 'User_X', 'User_Y', 'User_Z' for user and 'AGV_X', 'AGV_Y', 'AGV_Z' for AGV)
            user_dist = np.sqrt((last_row['User_X'] - first_row['User_X']) ** 2 +
                                (last_row['User_Y'] - first_row['User_Y']) ** 2 +
                                (last_row['User_Z'] - first_row['User_Z']) ** 2)
            
            agv_dist = np.sqrt((last_row['AGV_X'] - first_row['AGV_X']) ** 2 +
                               (last_row['AGV_Y'] - first_row['AGV_Y']) ** 2 +
                               (last_row['AGV_Z'] - first_row['AGV_Z']) ** 2)
            
            # Calculate the relative speeds
            user_relative_speed = user_dist / time_diff
            agv_relative_speed = agv_dist / time_diff
            
            # Update self.per_interaction_data with the calculated speeds
            self.per_interaction_data.loc[(self.per_interaction_data['PID'] == pid) & 
                                          (self.per_interaction_data['DRate'] == drate) &
                                          (self.per_interaction_data['AGVname'] == agvname),
                                          ['User_Relative_Speed', 'AGV_Relative_Speed']] = [user_relative_speed, agv_relative_speed]


    

    def calculate_frechet_distance(self):
        sys.setrecursionlimit(50000)
        time_windows = [3, 5, 7, 10]
        """Calculate the Fréchet distance for each unique PID, AGVname, and DRate across multiple time windows."""
        '''
        # Filter study_data for specific 'PID', 'AGVname', and 'DRate'
        filtered_data = study_data[(study_data['PID'] == '002') & 
                                   (study_data['AGVname'] == 1) & 
                                   (study_data['DRate'] == 'High')]
        '''
        # Ensure the 'Timestamp' column is in datetime format
        study_data['Timestamp'] = pd.to_datetime(study_data['Timestamp'])
        
        # Group by the filtered data 'PID', 'AGVname', and 'DRate
        grouped = study_data.groupby(['PID', 'AGVname', 'DRate'])
    
        # Create a dictionary to store results
        frechet_results = {time_window: {} for time_window in time_windows}
    
        for name, group in grouped:
            # Find interaction point (minimum AGV_User_distance)
            interaction_point = group.loc[group['AGV_User_distance'].idxmin()]
            interaction_time = interaction_point['Timestamp']
            
            # Iterate over each time window
            for time_window in time_windows:
                # Define time window
                start_time = interaction_time - pd.Timedelta(seconds=time_window)
                end_time = interaction_time + pd.Timedelta(seconds=time_window)
                
                # Filter data within time window
                window_data = group[(group['Timestamp'] >= start_time) & (group['Timestamp'] <= end_time)]
    
                # Check if there is enough data to compute the trajectory
                if len(window_data) < 2:
                    continue  # Not enough data to calculate the trajectory
    
                # Actual trajectory
                actual_path = window_data[['User_X', 'User_Y']].values
    
                # Generate expected trajectory points from the start to the end
                expected_start_point = actual_path[0]
                expected_end_point = actual_path[-1]
                expected_path = generate_expected_trajectory(expected_start_point, expected_end_point, num_points=len(actual_path))
    
                # Calculate the Fréchet distance
                frechet_dist = frechet_distance(expected_path, actual_path)
    
                # Round the Fréchet distance to two significant figures
                frechet_dist = round(frechet_dist, 2)
    
                # Store the result in the dictionary for the current time window
                frechet_results[time_window][name] = frechet_dist
    
        # Map results back to per_interaction_data for each time window for the purpose of debugging
        for time_window, result_dict in frechet_results.items():
            column_name = f'Frechet_Distance_{time_window}'
            for (pid, agvname, drate), distance in result_dict.items():
                per_interaction_data.loc[
                    (per_interaction_data['PID'] == pid) & 
                    (per_interaction_data['AGVname'] == agvname) & 
                    (per_interaction_data['DRate'] == drate), 
                    column_name
                ] = distance
                print(f"PID: {pid}, AGVname: {agvname}, DRate: {drate}")
    
        # Return the updated per_interaction_data with the calculated distances
        # return self.per_interaction_data
    
    def generate_expected_trajectory(start: tuple, end: tuple, num_points: int) -> np.ndarray:
        """Generate expected trajectory points from the start to the end."""
        return np.column_stack((
            np.linspace(start[0], end[0], num_points),
            np.linspace(start[1], end[1], num_points)
        ))
    
    def frechet_distance(P: np.ndarray, Q: np.ndarray) -> float:
        """Calculate the Fréchet distance between two trajectories."""
        ca = np.full((len(P), len(Q)), -1.0)
        return _c(ca, P, Q, len(P) - 1, len(Q) - 1)
    
    def _c(ca: np.ndarray, P: np.ndarray, Q: np.ndarray, i: int, j: int) -> float:
        """Recursive helper function for calculating Fréchet distance."""
        if ca[i, j] > -1:
            return ca[i, j]
        elif i == 0 and j == 0:
            ca[i, j] = euclidean_distance(P[0:1], Q[0:1])
        elif i > 0 and j == 0:
            ca[i, j] = max(_c(ca, P, Q, i - 1, 0), euclidean_distance(P[i:i + 1], Q[0:1]))
        elif i == 0 and j > 0:
            ca[i, j] = max(_c(ca, P, Q, 0, j - 1), euclidean_distance(P[0:1], Q[j:j + 1]))
        elif i > 0 and j > 0:
            ca[i, j] = max(
                min(
                    _c(ca, P, Q, i - 1, j),
                    _c(ca, P, Q, i - 1, j - 1),
                    _c(ca, P, Q, i, j - 1)
                ),
                euclidean_distance(P[i:i + 1], Q[j:j + 1])
            )
        else:
            ca[i, j] = float('inf')
        return ca[i, j]
    
    def euclidean_distance(p1: np.ndarray, p2: np.ndarray) -> float:
        """Calculate Euclidean distance between two points."""
        return np.linalg.norm(p1 - p2, axis=1)[0]

    
    def process(self, high_first_low_last, low_first_high_last, cross_first):
        """Run all processing steps for per-interaction data."""
        self.swap_agvname_values()
        self.drop_pid_rows()
        self.apply_zero_padding_to_pid()
        self.calculate_gaze_on_agv()
        self.map_user_trajectory()
        self.map_agv_approaching()
        self.create_agv_user_combination()
        self.map_agv_path_complexity()
        self.calculate_trust_before(high_first_low_last, low_first_high_last)  
        self.process_cross_first(cross_first) 
        self.calculate_relative_speed()
        self.calculate_frechet_distance()
        return self.per_interaction_data